# VC — Práctica 3 · Tarea 2 — Clasificación de microplásticos

**FRA / PEL / TAR** mediante segmentación clásica OpenCV, descriptores geométricos y de apariencia y aprendizaje automático **tradicional**. Sin aprendizaje profundo.

Se utilizan las fotografías `.JPG` completas para entrenamiento, ya que aportan más ejemplos. El test y `MPs_test_bbs.csv` quedan **reservados exclusivamente a la evaluación final**. No se optimizan umbrales, modelos ni pesos a partir del test.

In [ ]:
import cv2, csv, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score
CLASES = ['FRA','PEL','TAR']
ARCHIVOS = {
    'FRA':'fragment-03-olympus-10-01-2020.JPG',
    'PEL':'pellet-03-olympus-10-01-2020.JPG',
    'TAR':'tar-03-olympus-10-01-2020.JPG'
}
for archivo in [*ARCHIVOS.values(),'MPs_test.jpg','MPs_test_bbs.csv']:
    if not Path(archivo).exists(): raise FileNotFoundError(archivo)


## 1. Segmentación de imágenes de entrenamiento

Las fotografías completas tienen fondo casi uniforme. Utilizamos una diferencia respecto a la mediana de brillo de cada imagen, con umbrales definidos para cada clase **solo a partir de las fotografías de entrenamiento**. Para eliminar falsos positivos se filtra por área. La segmentación del test será diferente porque su fondo presenta bandas e iluminación desigual. No se utiliza el CSV para ajustar estos parámetros.

In [ ]:
# Parámetros específicos de las imágenes de entrenamiento (NO usar etiquetas de test)
PARAM_ENTRENO = {'FRA': (20, 30), 'PEL': (30, 60), 'TAR': (30, 60)}
MARGEN = 10

def obtener_contornos(mask, area_min, area_max, margen=MARGEN):
    h, w = mask.shape
    n, lab, datos, _ = cv2.connectedComponentsWithStats(mask, 8)
    limpia = np.zeros_like(mask)
    for k in range(1,n):
        x,y,an,al,area = datos[k]
        if area < area_min or area > area_max: continue
        if x <= margen or y <= margen or x+an >= w-margen or y+al >= h-margen: continue
        limpia[lab==k]=255
    cont, _ = cv2.findContours(limpia, cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    return limpia,cont

def segmentar_entrenamiento(im, clase):
    gris = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    umbral, area_min = PARAM_ENTRENO[clase]
    ref = float(np.median(gris))
    mascara = np.uint8(gris < ref - umbral)*255
    return obtener_contornos(mascara, area_min, 5000)

def segmentar_test(im):
    # Fondo local (no se usa el CSV de anotaciones)
    lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB).astype(np.float32)
    fondo = cv2.GaussianBlur(lab, (0,0),24)
    delta = np.abs(lab-fondo)
    mascara = np.uint8((delta[:,:,0]>10) | (np.max(delta[:,:,1:],axis=2)>11))*255
    # Rama adicional de objetos muy oscuros sobre el fondo claro.
    # Ayuda a recuperar TAR cuya zona central no quedó completa.
    gris=cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)
    oscuros=np.uint8(gris<135)*255
    mascara=cv2.bitwise_or(mascara,oscuros)
    mascara=cv2.morphologyEx(mascara,cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))
    # Se filtran los contornos sin watershed: no se dividen partículas artificialmente.
    return obtener_contornos(mascara, 18, 5000, margen=9)

def mostrar(im, mask, cont, titulo):
    copia=im.copy()
    cv2.drawContours(copia,cont,-1,(0,255,0),2)
    fig,ax=plt.subplots(1,3,figsize=(14,5))
    ax[0].imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB))
    ax[1].imshow(mask,cmap='gray')
    ax[2].imshow(cv2.cvtColor(copia,cv2.COLOR_BGR2RGB))
    for a,t in zip(ax,['Original','Máscara','Contornos']):
        a.set_title(t);a.axis('off')
    fig.suptitle(f'{titulo}: {len(cont)} detecciones')
    plt.tight_layout();plt.show()

for clase,ruta in ARCHIVOS.items():
    im=cv2.imread(ruta)
    mascara,cs=segmentar_entrenamiento(im,clase)
    print(f'{clase}: {len(cs)} partículas de entrenamiento')
    mostrar(im,mascara,cs,clase)


## 2. Características geométricas y apariencia relativa al fondo

Se utilizan circularidad, solidez, ocupación del rectángulo, razón de ejes, compacidad, contraste normalizado respecto al fondo y color relativo. La normalización relativa evita comparar directamente la iluminación de fotografías tomadas con diferentes condiciones. No se usa el test para ajustar pesos.


In [ ]:
def extraer(im, cont):
    area = cv2.contourArea(cont)
    per = cv2.arcLength(cont, True)
    if area < 1 or per < 1:
        return None
    x,y,w,h = cv2.boundingRect(cont)
    circularidad = np.clip(4*np.pi*area/(per*per),0,1)
    solidez = area/max(cv2.contourArea(cv2.convexHull(cont)),1)
    ocupacion = area/(w*h)
    proporcion = min(w,h)/max(w,h)
    eje = proporcion
    if len(cont)>=5:
        (_, _),(e1,e2),_ = cv2.fitEllipse(cont)
        if max(e1,e2)>0: eje=min(e1,e2)/max(e1,e2)
    # Medidas de color y brillo *locales*, sin usar intensidades absolutas de una foto
    h_im,w_im=im.shape[:2]
    borde = max(7, int(max(w,h)*0.5))
    x0=max(0,x-borde); y0=max(0,y-borde)
    x1=min(w_im,x+w+borde); y1=min(h_im,y+h+borde)
    roi=im[y0:y1,x0:x1]
    mask=np.zeros((y1-y0,x1-x0),np.uint8)
    cv2.drawContours(mask,[cont-np.array([[[x0,y0]]])],-1,255,-1)
    hsv=cv2.cvtColor(roi,cv2.COLOR_BGR2HSV)
    lab=cv2.cvtColor(roi,cv2.COLOR_BGR2LAB)
    entorno=cv2.dilate(mask,np.ones((9,9),np.uint8))
    entorno=cv2.dilate(entorno,np.ones((max(11,borde//2*2+1),)*2,np.uint8))
    fondo=cv2.bitwise_and(cv2.bitwise_not(mask),entorno)
    # Si hay muy pocos píxeles del entorno, recurrir a todo el recorte exterior
    if cv2.countNonZero(fondo)<20: fondo=cv2.bitwise_not(mask)
    Lobj=float(np.median(lab[:,:,0][mask>0]))
    Lfon=float(np.median(lab[:,:,0][fondo>0]))
    diferencia=(Lobj-Lfon)/max(Lfon,1)
    Sobj=float(np.median(hsv[:,:,1][mask>0]))/255
    Sfon=float(np.median(hsv[:,:,1][fondo>0]))/255
    Aobj=float(np.median(lab[:,:,1][mask>0]))
    Bobj=float(np.median(lab[:,:,2][mask>0]))
    Afon=float(np.median(lab[:,:,1][fondo>0]))
    Bfon=float(np.median(lab[:,:,2][fondo>0]))
    # Fracción de píxeles internos significativamente más oscuros que su fondo
    oscuro=float(np.mean(lab[:,:,0][mask>0] < Lfon*0.65))
    # No se incluye tamaño: fotografías con escalas muy diferentes
    return [circularidad,solidez,ocupacion,proporcion,eje,diferencia,
            Sobj-Sfon,(Aobj-Afon)/64,(Bobj-Bfon)/64,oscuro]


## 3. Selección del clasificador mediante validación en entrenamiento

Probamos tres técnicas tradicionales: k-NN, SVM y Random Forest. Elegimos la que consigue mayor **F1 macro** en validación cruzada estratificada con las muestras de entrenamiento (evita escoger un método por favorecer únicamente a FRA). La selección **no utiliza el CSV del test**.

La validación tiene una limitación: todas las partículas de cada clase proceden de una misma fotografía; por eso la puntuación de validación puede resultar optimista respecto a fotografías nuevas.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score

X, Y = [], []
for clase, ruta in ARCHIVOS.items():
    imagen = cv2.imread(ruta)
    _, contornos_entreno = segmentar_entrenamiento(imagen, clase)
    for c in contornos_entreno:
        v = extraer(imagen, c)
        if v is not None and np.all(np.isfinite(v)):
            X.append(v)
            Y.append(clase)
X = np.asarray(X, dtype=float)
Y = np.asarray(Y)
print('Partículas de entrenamiento:', {s:int(sum(Y==s)) for s in CLASES})

modelos = {
    'k-NN': make_pipeline(SimpleImputer(), StandardScaler(), KNeighborsClassifier(n_neighbors=5, weights='distance')),
    'SVM-RBF': make_pipeline(SimpleImputer(), StandardScaler(), SVC(C=2, gamma='scale', class_weight='balanced')),
    'Random Forest': make_pipeline(SimpleImputer(), RandomForestClassifier(n_estimators=240, max_features='sqrt', min_samples_leaf=2, class_weight='balanced', random_state=33)),
    'Extra Trees': make_pipeline(SimpleImputer(), ExtraTreesClassifier(n_estimators=240, min_samples_leaf=2, class_weight='balanced', random_state=33)),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=33)
resultados_cv = {}
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X, Y, scoring='f1_macro', cv=cv)
    resultados_cv[nombre] = float(scores.mean())
    print(f'{nombre:14s} F1 macro validación: {scores.mean():.3f} ± {scores.std():.3f}')

mejor_nombre = max(resultados_cv, key=resultados_cv.get)
clasificador = modelos[mejor_nombre]
clasificador.fit(X, Y)
print('Modelo elegido (solo entrenamiento):', mejor_nombre)

def predecir_clase(vector):
    return str(clasificador.predict(np.asarray(vector).reshape(1, -1))[0])

## 4. Aplicación al test

No se consulta `MPs_test_bbs.csv` para segmentar ni clasificar. Se muestra la máscara y la clasificación. Los errores visibles deben comentarse en la entrega.

In [ ]:
im=cv2.imread('MPs_test.jpg')
mask,contornos=segmentar_test(im)
mostrar(im,mask,contornos,'Test sin anotaciones')
predicciones=[]
vista=im.copy()
colores={'FRA':(0,0,255),'PEL':(0,255,0),'TAR':(255,0,0)}
for ct in contornos:
    v=extraer(im,ct)
    if v is None:continue
    clase=predecir_clase(v)
    x,y,w,h=cv2.boundingRect(ct)
    predicciones.append({'label':clase,'bbox':(x,y,x+w,y+h)})
    cv2.drawContours(vista,[ct],-1,colores[clase],2)
    cv2.putText(vista,clase,(x,max(15,y-5)),cv2.FONT_HERSHEY_SIMPLEX,.42,colores[clase],1)
print('Predicciones:',len(predicciones),{s:sum(p['label']==s for p in predicciones) for s in CLASES})
plt.figure(figsize=(10,12));plt.imshow(cv2.cvtColor(vista,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()


## 5. Evaluación con anotaciones del test

Este es el **único apartado** que lee `MPs_test_bbs.csv`. Las métricas de clasificación se calculan sobre partículas emparejadas individualmente por IoU; por eso se muestra también cuántas anotaciones quedaron sin detectar y cuántas detecciones no pudieron emparejarse.


In [ ]:
def iou(a,b):
    x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    bb=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/max(aa+bb-inter,1e-6)

anotaciones=[]
with open('MPs_test_bbs.csv',newline='',encoding='utf-8-sig') as f:
    for fila in csv.DictReader(f):
        clase=fila['label'].strip().upper()
        if clase in CLASES:
            caja=tuple(int(float(fila[z])) for z in ['x_min','y_min','x_max','y_max'])
            anotaciones.append({'label':clase,'bbox':caja})

UMBRAL_IOU=0.2
posibles=[]
for i,a in enumerate(anotaciones):
    for j,p in enumerate(predicciones):
        solape=iou(a['bbox'],p['bbox'])
        if solape>=UMBRAL_IOU:posibles.append((solape,i,j))
usadas_a=set();usadas_p=set();parejas=[]
for score,i,j in sorted(posibles,reverse=True):
    if i not in usadas_a and j not in usadas_p:
        usadas_a.add(i);usadas_p.add(j);parejas.append((i,j))
real=[anotaciones[i]['label'] for i,j in parejas]
pred=[predicciones[j]['label'] for i,j in parejas]
print('Anotaciones:',len(anotaciones),'Predicciones:',len(predicciones))
print('Emparejadas:',len(parejas),'No detectadas:',len(anotaciones)-len(parejas))
print('Detecciones sin anotación:',len(predicciones)-len(parejas))
print('Cobertura:',round(len(parejas)/len(anotaciones),3) if anotaciones else 0)
if real:
    matriz=confusion_matrix(real,pred,labels=CLASES)
    ConfusionMatrixDisplay(matriz,display_labels=CLASES).plot(cmap='Blues',values_format='d')
    plt.title('Matriz de confusión (partículas emparejadas)');plt.show()
    print(classification_report(real,pred,labels=CLASES,zero_division=0,digits=3))
    print('Accuracy:',accuracy_score(real,pred))
else:
    print('No se pudo emparejar ninguna partícula. Revisar las máscaras.')

## 6. Conclusiones y límites

- **Método:** detección de partículas por umbrales y contornos; representación con circularidad, solidez, proporciones geométricas y diferencias de apariencia respecto al fondo local; clasificación supervisada tradicional.
- **Imágenes:** fotografías completas de FRA, PEL y TAR para obtener ejemplos; `MPs_test.jpg` solo se procesa al final y su CSV solo se abre en evaluación.
- **Resultado:** interpretar matriz de confusión, F1 macro y cobertura de detección **conjuntamente**. La accuracy sobre partículas emparejadas no mide los objetos omitidos ni falsos positivos.
- **Limitaciones:** variaciones de iluminación, objetos claros sobre fondos claros, tamaños y escalas distintos, y similitud visual entre pellets oscuros y alquitrán. La selección por validación cruzada en fotos individuales no garantiza generalización.

Para una entrega reproducible conservar imágenes originales y notebook en la misma carpeta; especificar versión de Python y paquetes (`opencv-python`, `numpy`, `matplotlib`, `scikit-learn`).